1. Data upload 

In [2]:
import kagglehub
import numpy as np

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

/Users/annabieliaieva/Desktop/Projects/Prices/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: /Users/annabieliaieva/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques


In [3]:
import os

print(os.listdir(path))

['test.csv', 'data_description.txt', 'train.csv', 'sample_submission.csv']


In [4]:
import pandas as pd

train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")

train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
description = open(f"{path}/data_description.txt")
description.read()

"MSSubClass: Identifies the type of dwelling involved in the sale.\t\n\n        20\t1-STORY 1946 & NEWER ALL STYLES\n        30\t1-STORY 1945 & OLDER\n        40\t1-STORY W/FINISHED ATTIC ALL AGES\n        45\t1-1/2 STORY - UNFINISHED ALL AGES\n        50\t1-1/2 STORY FINISHED ALL AGES\n        60\t2-STORY 1946 & NEWER\n        70\t2-STORY 1945 & OLDER\n        75\t2-1/2 STORY ALL AGES\n        80\tSPLIT OR MULTI-LEVEL\n        85\tSPLIT FOYER\n        90\tDUPLEX - ALL STYLES AND AGES\n       120\t1-STORY PUD (Planned Unit Development) - 1946 & NEWER\n       150\t1-1/2 STORY PUD - ALL AGES\n       160\t2-STORY PUD - 1946 & NEWER\n       180\tPUD - MULTILEVEL - INCL SPLIT LEV/FOYER\n       190\t2 FAMILY CONVERSION - ALL STYLES AND AGES\n\nMSZoning: Identifies the general zoning classification of the sale.\n\t\t\n       A\tAgriculture\n       C\tCommercial\n       FV\tFloating Village Residential\n       I\tIndustrial\n       RH\tResidential High Density\n       RL\tResidential Low Densi

2. Data Preprocessing

Missing-value treatment

Missing values are handled according to their meaning. Values that indicate the absence of a property feature are replaced with `"None"` or `0`, while unknown values are filled using an appropriate statistic such as the median or mode.


In [6]:
def handle_missing_values(train_df, test_df):
    """
    Fill missing values according to their meaning.

    Statistics used for imputation are calculated only from the
    training dataset and then applied to both datasets.
    """

    train_df = train_df.copy()
    test_df = test_df.copy()

    # 1. Missing categorical value means feature is absent

    absence_categorical_cols = [
        "Alley",
        "MasVnrType",
        "BsmtQual",
        "BsmtCond",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
        "FireplaceQu",
        "GarageType",
        "GarageFinish",
        "GarageQual",
        "GarageCond",
        "PoolQC",
        "Fence",
        "MiscFeature"
    ]

    for col in absence_categorical_cols:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna("None")
            test_df[col] = test_df[col].fillna("None")

    # 2. Missing numerical value means feature is absent

    absence_numeric_cols = [
        "MasVnrArea",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "BsmtFullBath",
        "BsmtHalfBath",
        "GarageCars",
        "GarageArea"
    ]

    for col in absence_numeric_cols:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)

    # 3. LotFrontage: neighborhood median
    # LotFrontage represents a real measurement, so zero would imply
    # that the property has no street frontage. Missing values are
    # therefore filled with the median frontage of the property's
    # neighborhood. The overall training median is used as a fallback.

    neighborhood_frontage = (
        train_df.groupby("Neighborhood")["LotFrontage"].median()
    )

    overall_frontage = train_df["LotFrontage"].median()

    train_df["LotFrontage"] = (
        train_df["LotFrontage"]
        .fillna(train_df["Neighborhood"].map(neighborhood_frontage))
        .fillna(overall_frontage)
    )

    test_df["LotFrontage"] = (
        test_df["LotFrontage"]
        .fillna(test_df["Neighborhood"].map(neighborhood_frontage))
        .fillna(overall_frontage)
    )

    # 4. Create garage-presence feature 

    train_df["HasGarage"] = (
        train_df["GarageType"].ne("None").astype(int)
    )

    test_df["HasGarage"] = (
        test_df["GarageType"].ne("None").astype(int)
    )
    # 5. Unknown categorical values: use training mode

    unknown_categorical_cols = [
        "MSZoning",
        "Utilities",
        "Exterior1st",
        "Exterior2nd",
        "Electrical",
        "KitchenQual",
        "Functional",
        "SaleType"
    ]

    for col in unknown_categorical_cols:
        if col in train_df.columns:
            most_frequent = train_df[col].mode(dropna=True)[0]

            train_df[col] = train_df[col].fillna(most_frequent)
            test_df[col] = test_df[col].fillna(most_frequent)

    return train_df, test_df

train, test = handle_missing_values(train, test)

In [7]:
# Fill only genuinely unknown garage years for houses
# that are recorded as having a garage.
garage_year_median = train.loc[
    train["HasGarage"] == 1,
    "GarageYrBlt"
].median()

train.loc[
    (train["HasGarage"] == 1)
    & (train["GarageYrBlt"].isna()),
    "GarageYrBlt"
] = garage_year_median

test.loc[
    (test["HasGarage"] == 1)
    & (test["GarageYrBlt"].isna()),
    "GarageYrBlt"
] = garage_year_median

# Calculate garage age; houses without garages receive age 0.
train["GarageAge"] = np.where(
    train["HasGarage"] == 1,
    train["YrSold"] - train["GarageYrBlt"],
    0
)

test["GarageAge"] = np.where(
    test["HasGarage"] == 1,
    test["YrSold"] - test["GarageYrBlt"],
    0
)

train["GarageAge"] = train["GarageAge"].clip(lower=0)
test["GarageAge"] = test["GarageAge"].clip(lower=0)

# Remove the original year after extracting the useful age feature.
train.drop(columns="GarageYrBlt", inplace=True)
test.drop(columns="GarageYrBlt", inplace=True)

In [8]:
# Check remaining missing values after imputation
missing_after_imputation_train = train.isnull().sum()
missing_after_imputation_train = missing_after_imputation_train[
    missing_after_imputation_train > 0
]

print(
    f"Columns with missing values after imputation (train set): "
    f"{len(missing_after_imputation_train)}"
)

missing_after_imputation_test = test.isnull().sum()
missing_after_imputation_test = missing_after_imputation_test[
    missing_after_imputation_test > 0
]

print(
    f"Columns with missing values after imputation (test set): "
    f"{len(missing_after_imputation_test)}"
)

Columns with missing values after imputation (train set): 0
Columns with missing values after imputation (test set): 0


In [9]:
# Encode ordinal categorical features by replacing
# ordered category labels with numerical values
# while preserving their ranking relationships.

ordinal_mappings = {

    'ExterQual': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'ExterCond': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'KitchenQual': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'HeatingQC': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'GarageQual': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'GarageCond': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'BsmtQual': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'BsmtCond': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'FireplaceQu': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'PoolQC': {
        'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5
    },

    'BsmtExposure': {
        'None':0, 'No':1, 'Mn':2, 'Av':3, 'Gd':4
    },

    'BsmtFinType1': {
        'None':0, 'Unf':1, 'LwQ':2,
        'Rec':3, 'BLQ':4,
        'ALQ':5, 'GLQ':6
    },

    'BsmtFinType2': {
        'None':0, 'Unf':1, 'LwQ':2,
        'Rec':3, 'BLQ':4,
        'ALQ':5, 'GLQ':6
    },

    'GarageFinish': {
        'None':0,
        'Unf':1,
        'RFn':2,
        'Fin':3
    }
}

def encode_ordinal_features(df, mappings):
    """
    Encodes ordinal categorical features using predefined mappings.

    Parameters:
    df (DataFrame): Input dataset
    mappings (dict): Dictionary containing feature mappings

    Returns:
    DataFrame: Dataset with encoded ordinal features
    """
    df = df.copy()

    for column, mapping in mappings.items():
        unknown_values = (
            set(df[column].dropna().unique()) - set(mapping.keys())
        )

        if unknown_values:
            raise ValueError(
                f"Unexpected values in {column}: {unknown_values}"
            )

        df[column] = df[column].map(mapping)

    return df


# Apply encoding to both training and test datasets
train = encode_ordinal_features(
    train,
    ordinal_mappings
)

test = encode_ordinal_features(
    test,
    ordinal_mappings
)

# Verify that encoding did not introduce unexpected missing values
missing_after_encoding_train = train.isnull().sum()
missing_after_encoding_train = missing_after_encoding_train[
    missing_after_encoding_train > 0
]

missing_after_encoding_test = test.isnull().sum()
missing_after_encoding_test = missing_after_encoding_test[
    missing_after_encoding_test > 0
]

print(
    f"Train columns with missing values after encoding: "
    f"{len(missing_after_encoding_train)}"
)

print(
    f"Test columns with missing values after encoding: "
    f"{len(missing_after_encoding_test)}"
)

Train columns with missing values after encoding: 0
Test columns with missing values after encoding: 0


In [10]:
nominal_columns = train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print(f"Number of remaining nominal columns: {len(nominal_columns)}")

for column in nominal_columns:
    print(
        f"{column}: "
        f"{train[column].nunique(dropna=False)} unique values"
    )

Number of remaining nominal columns: 29
MSZoning: 5 unique values
Street: 2 unique values
Alley: 3 unique values
LotShape: 4 unique values
LandContour: 4 unique values
Utilities: 2 unique values
LotConfig: 5 unique values
LandSlope: 3 unique values
Neighborhood: 25 unique values
Condition1: 9 unique values
Condition2: 8 unique values
BldgType: 5 unique values
HouseStyle: 8 unique values
RoofStyle: 6 unique values
RoofMatl: 8 unique values
Exterior1st: 15 unique values
Exterior2nd: 16 unique values
MasVnrType: 4 unique values
Foundation: 6 unique values
Heating: 6 unique values
CentralAir: 2 unique values
Electrical: 5 unique values
Functional: 7 unique values
GarageType: 7 unique values
PavedDrive: 3 unique values
Fence: 5 unique values
MiscFeature: 5 unique values
SaleType: 9 unique values
SaleCondition: 6 unique values


In [20]:
ordinal_columns = list(ordinal_mappings.keys())

for column in ordinal_columns:
    print(
        column,
        train[column].dtype,
        train[column].unique()
    )

ExterQual int64 [4 3 5 2]
ExterCond int64 [3 4 2 1 5]
KitchenQual int64 [4 3 5 2]
HeatingQC int64 [5 4 3 2 1]
GarageQual int64 [3 2 4 0 5 1]
GarageCond int64 [3 2 0 4 1 5]
BsmtQual int64 [4 3 5 0 2]
BsmtCond int64 [3 4 0 2 1]
FireplaceQu int64 [0 3 4 2 5 1]
PoolQC int64 [0 5 2 4]
BsmtExposure int64 [1 4 2 3 0]
BsmtFinType1 int64 [6 5 1 3 4 0 2]
BsmtFinType2 int64 [1 4 0 5 3 2 6]
GarageFinish int64 [2 1 3 0]


In [11]:
# Validate missing-value treatment and ordinal encoding
assert train.isnull().sum().sum() == 0, \
    "Missing values detected in training data"

assert test.isnull().sum().sum() == 0, \
    "Missing values detected in test data"

print("Preprocessing completed successfully")

Preprocessing completed successfully


In [12]:
# Some numerical variables represent categories rather than continuous values.
# Convert them into categorical type for better interpretation.

train['MSSubClass'] = train['MSSubClass'].astype('category')
test['MSSubClass'] = test['MSSubClass'].astype('category')

train['MoSold'] = train['MoSold'].astype('category')
test['MoSold'] = test['MoSold'].astype('category')

In [13]:
# Create age-based features while keeping the original year columns
# for descriptive analysis and data-quality checks.

train["HouseAge"] = (
    train["YrSold"] - train["YearBuilt"]
).clip(lower=0)

test["HouseAge"] = (
    test["YrSold"] - test["YearBuilt"]
).clip(lower=0)

train["RemodAge"] = (
    train["YrSold"] - train["YearRemodAdd"]
).clip(lower=0)

test["RemodAge"] = (
    test["YrSold"] - test["YearRemodAdd"]
).clip(lower=0)



In [14]:
# Area features can be combined to create a new feature that 
# represents the total area of the house.
train['TotalArea'] = (
    train['TotalBsmtSF'] +
    train['1stFlrSF'] +
    train['2ndFlrSF'] +
    train['GarageArea']
)

test['TotalArea'] = (
    test['TotalBsmtSF'] +
    test['1stFlrSF'] +
    test['2ndFlrSF'] +
    test['GarageArea']
)

In [15]:
# Validation
# Confirm that feature engineering did not introduce missing
# values or logically invalid age features.

assert (train["HouseAge"] >= 0).all(), \
    "Negative HouseAge detected in training data"

assert (test["HouseAge"] >= 0).all(), \
    "Negative HouseAge detected in test data"

assert (train["RemodAge"] >= 0).all(), \
    "Negative RemodAge detected in training data"

assert (test["RemodAge"] >= 0).all(), \
    "Negative RemodAge detected in test data"

assert (train["GarageAge"] >= 0).all(), \
    "Negative GarageAge detected in training data"

assert (test["GarageAge"] >= 0).all(), \
    "Negative GarageAge detected in test data"

print("Preprocessing completed successfully")

Preprocessing completed successfully


In [16]:
# Create binary indicators for optional property features.
# A value of 1 means that the feature is present; 0 means absent.

train["HasPool"] = (train["PoolArea"] > 0).astype(int)
test["HasPool"] = (test["PoolArea"] > 0).astype(int)

train["HasFireplace"] = (
    train["Fireplaces"] > 0
).astype(int)
test["HasFireplace"] = (
    test["Fireplaces"] > 0
).astype(int)

train["HasBsmt"] = (
    train["TotalBsmtSF"] > 0
).astype(int)
test["HasBsmt"] = (
    test["TotalBsmtSF"] > 0
).astype(int)

train["HasPorch"] = (
    (train["OpenPorchSF"] > 0)
    | (train["EnclosedPorch"] > 0)
    | (train["3SsnPorch"] > 0)
    | (train["ScreenPorch"] > 0)
).astype(int)

test["HasPorch"] = (
    (test["OpenPorchSF"] > 0)
    | (test["EnclosedPorch"] > 0)
    | (test["3SsnPorch"] > 0)
    | (test["ScreenPorch"] > 0)
).astype(int)

In [17]:
#We can also create a new feature that represents the total number of extras a house has. 
#This can be calculated as the sum of the binary features we created above.

train['TotalExtras'] = (
    train['HasPool'] +
    train['HasGarage'] +
    train['HasFireplace'] +
    train['HasPorch'] +
    train['HasBsmt']
)

test['TotalExtras'] = (
    test['HasPool'] +
    test['HasGarage'] +
    test['HasFireplace'] +
    test['HasPorch'] +
    test['HasBsmt']
)

In [18]:
# What's more we can create a new feature that represents the total number of bathrooms in a house.
train['TotalBathrooms'] = (
    train['FullBath'] +
    (0.5 * train['HalfBath']) +
    train['BsmtFullBath'] +
    (0.5 * train['BsmtHalfBath'])
)
test['TotalBathrooms'] = (
    test['FullBath'] +                       
    (0.5 * test['HalfBath']) +
    test['BsmtFullBath'] +
    (0.5 * test['BsmtHalfBath'])
)             

In [19]:
# Apply logarithmic transformation to the target variable.
# House prices are typically right-skewed, and log transformation
# helps reduce skewness and stabilize variance.
train['SalePrice'] = np.log1p(train['SalePrice'])

## Data preprocessing summary

The preprocessing stage prepared the Ames Housing data for exploratory analysis and predictive modeling.

* Missing numerical values were replaced with `0` where a missing value represents the absence of a property feature, such as a garage, basement, pool, or masonry veneer.
* Missing categorical values were replaced with `"None"` where the missing value represents the absence of that feature.
* Ordered categorical variables, including property-quality and condition ratings, were converted to numerical scores using predefined ordinal mappings.
* Numerical codes that represent categories rather than measurable quantities, such as `MSSubClass` and `MoSold`, were converted to categorical variables.
* New features were engineered to summarize property age, size, facilities, and amenities:

  * `HouseAge`
  * `RemodAge`
  * `TotalArea`
  * `TotalBathrooms`
  * `HasPool`
  * `HasGarage`
  * `HasFireplace`
  * `HasBsmt`
  * `HasPorch`
  * `TotalExtras`
* Data-quality checks were used to identify inconsistent combinations, such as properties with garage area but no recorded garage.
* The original `SalePrice` variable was preserved, while a separate `LogSalePrice` target was created to reduce skewness for regression modeling.
* Remaining nominal categorical variables will be one-hot encoded inside the modeling pipeline.
* Scaling, encoding, and other learned transformations will be fitted only on the training folds during cross-validation to prevent information leakage.
